In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


df = pd.read_csv('/kaggle/input/datasets/shauryapandey19/spatiotemp/delhi_20station_pm25_spatiotemporal_dataset_final.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.drop_duplicates(['station_name', 'timestamp'])

stations = sorted(df['station_name'].unique())
S = len(stations)

def wide(col):
    # hours x stations table, NaN where a station has no row at that hour
    return df.pivot(index='timestamp', columns='station_name', values=col)[stations].to_numpy()

PM  = wide('pm25')            # neighbour values 
PMO = wide('pm25_original')   # targets (same as IDW baseline)
SIN = wide('wind_dir_sin')
COS = wide('wind_dir_cos')
SPD = wide('wind_speed_m_s')

# ---------- station geometry ----------
coords = (df.drop_duplicates('station_name')
            .set_index('station_name')
            .loc[stations, ['latitude', 'longitude']])
lat = np.radians(coords['latitude'].to_numpy())
lon = np.radians(coords['longitude'].to_numpy())

dist = np.zeros((S, S))   # km
bear = np.zeros((S, S))   # compass bearing from station i to station j, radians
for i in range(S):
    for j in range(S):
        if i == j:
            continue
        dlat = lat[j] - lat[i]
        dlon = lon[j] - lon[i]
        a = np.sin(dlat / 2) ** 2 + np.cos(lat[i]) * np.cos(lat[j]) * np.sin(dlon / 2) ** 2
        dist[i, j] = 2 * 6371 * np.arcsin(np.sqrt(a))
        y = np.sin(dlon) * np.cos(lat[j])
        x = np.cos(lat[i]) * np.sin(lat[j]) - np.sin(lat[i]) * np.cos(lat[j]) * np.cos(dlon)
        bear[i, j] = np.arctan2(y, x)

# ---------- weighting schemes ----------
# d  = distance in km
# a  = alignment: +1 source directly upwind of target, -1 directly downwind
# tt = hours the wind needs to carry air from source to target
def w_idw(d, a, tt):
    return 1.0 / d ** 2

def w_wind_dir(d, a, tt):               # wind direction only, no distance term
    return 1e-3 + np.clip(a, 0, None) ** 2

def w_wind_travel(d, a, tt):            # direction + travel time (distance enters via d / speed)
    return 1e-3 + (np.clip(a, 0, None) ** 2) * np.exp(-tt / 3.0)

def w_idw_x_wind(d, a, tt):             # IDW boosted for upwind, damped for downwind
    return (1.0 / d ** 2) * (0.2 + 0.8 * (1 + a) / 2)

schemes = {
    'IDW (baseline)':      w_idw,
    'wind direction only': w_wind_dir,
    'wind + travel time':  w_wind_travel,
    'IDW x wind':          w_idw_x_wind,
}

# ---------- leave-one-station-out ----------
rows = {name: [] for name in schemes}

for j in range(S):
    src = [i for i in range(S) if i != j]

    P = PM[:, src]
    valid = ~np.isnan(P)
    P0 = np.where(valid, P, 0.0)

    # wind on the path = average of source and target wind vectors
    sin_m = (SIN[:, src] + SIN[:, [j]]) / 2
    cos_m = (COS[:, src] + COS[:, [j]]) / 2
    wind_to = np.arctan2(sin_m, cos_m) + np.pi            # direction the air is moving toward
    align = np.cos(wind_to - bear[src, j][None, :])       # +1 = source is upwind of target

    spd_m = np.maximum((SPD[:, src] + SPD[:, [j]]) / 2, 0.5)
    d = np.broadcast_to(dist[src, j][None, :], P.shape)
    tt = np.clip(d / (spd_m * 3.6), 0, 6)                 # km / (km/h) = hours

    # same rule as IDW loop: target has original PM2.5, at least 2 neighbours
    keep = ~np.isnan(PMO[:, j]) & (valid.sum(axis=1) >= 2)
    actual = PMO[keep, j]

    for name, fn in schemes.items():
        with np.errstate(invalid='ignore', divide='ignore'):
            w = np.where(valid, fn(d, align, tt), 0.0)     # zero weight for missing neighbours
            pred = (w * P0).sum(axis=1) / w.sum(axis=1)
        assert not np.isnan(pred[keep]).any()
        rows[name].append(pd.DataFrame({
            'station': stations[j],
            'actual': actual,
            'pred': pred[keep],
        }))

# ---------- results ----------
summary = []
per_station = {}
for name in schemes:
    r = pd.concat(rows[name], ignore_index=True)
    r['abs_err'] = (r['pred'] - r['actual']).abs()
    summary.append({
        'scheme': name,
        'n': len(r),
        'MAE': mean_absolute_error(r['actual'], r['pred']),
        'RMSE': mean_squared_error(r['actual'], r['pred']) ** 0.5,
        'R2': r2_score(r['actual'], r['pred']),
        'Bias': (r['pred'] - r['actual']).mean(),
    })
    per_station[name] = r.groupby('station')['abs_err'].mean()

print(pd.DataFrame(summary).round(3).to_string(index=False))
print()
print(pd.DataFrame(per_station).round(2))

             scheme      n    MAE   RMSE    R2   Bias
     IDW (baseline) 235876 22.600 39.555 0.778  1.557
wind direction only 235876 24.913 43.179 0.735 -0.003
 wind + travel time 235876 24.692 42.753 0.740  0.291
         IDW x wind 235876 22.821 39.884 0.774  1.625

                IDW (baseline)  wind direction only  wind + travel time  \
station                                                                   
alipur                   16.52                20.05               20.01   
anand_vihar              23.07                27.61               26.99   
aurobindo_marg           23.57                24.14               23.90   
aya_nagar                30.11                32.38               31.68   
bawana                   24.81                31.15               30.64   
dtu                      35.78                36.36               36.59   
mandir_marg              26.81                28.38               28.38   
mundka                   21.91                24.54   